# Final Comparison — Retrievers × Generators

## Purpose

Combines the final test-set evaluation results from all three generation experiments (`09_qwen_evaluation.ipynb`, `10_mt0_small_evaluation.ipynb`, and the failed `mt5_gold` attempt, kept for documentation) into a single comparison table and chart, for the project report.

## Included runs

- **Qwen3-8B**, concise prompt, test split — the chosen final Qwen configuration.
- **mT0-small**, test split — the successful seq2seq fine-tuned generator.
- **mT5-base (gold)**, validation split only — training failed 

In [10]:
import json
from pathlib import Path

import pandas as pd
import altair as alt

In [11]:
PROJECT_ROOT = Path.cwd()

EVALUATION_DIR = PROJECT_ROOT / "data" / "evaluation"

RESULT_FILES = {
    ("Qwen3-8B", "concise", "test"): EVALUATION_DIR / "qwen" / "qwen_test_concise_metrics.csv",
    ("mT0-small", "-", "test"): EVALUATION_DIR / "mt0_small" / "mt0_small_gold_v2_test_metrics.csv",
    ("mT5-base (gold)", "-", "validation"): EVALUATION_DIR / "mt5_gold" / "mt5_gold_validation_metrics.csv",
}

for key, path in RESULT_FILES.items():
    print(key, "->", path, "| postoji:" , path.exists())

('Qwen3-8B', 'concise', 'test') -> /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/qwen/qwen_test_concise_metrics.csv | postoji: True
('mT0-small', '-', 'test') -> /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/mt0_small/mt0_small_gold_v2_test_metrics.csv | postoji: True
('mT5-base (gold)', '-', 'validation') -> /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/mt5_gold/mt5_gold_validation_metrics.csv | postoji: True


In [12]:
comparison_frames = []

for (generator, prompt_variant, split), path in RESULT_FILES.items():
    if not path.exists():
        print(f"Preskočeno: {path} ne postoji.")
        continue

    df = pd.read_csv(path)

    df["generator"] = generator
    df["prompt_variant"] = prompt_variant
    df["split"] = split

    comparison_frames.append(df)

final_comparison_df = pd.concat(comparison_frames, ignore_index=True)

final_comparison_df

,retriever,EM,F1,ROUGE_L,Semantic_sim,generator,prompt_variant,split
0,dense,0.0,0.323050,0.278028,0.904510,Qwen3-8B,concise,test
1,tfidf,0.0,0.288707,0.233561,0.905888,Qwen3-8B,concise,test
2,bm25,0.0,0.288046,0.248253,0.903693,Qwen3-8B,concise,test
3,dense,0.0,0.171415,0.137277,0.856749,mT0-small,-,test
4,tfidf,0.0,0.149572,0.122563,0.861113,mT0-small,-,test
5,bm25,0.0,0.138575,0.112511,0.858987,mT0-small,-,test
6,tfidf,0.0,0.091717,0.073055,NaN,mT5-base (gold),-,validation
7,dense,0.0,0.079957,0.074005,NaN,mT5-base (gold),-,validation
8,bm25,0.0,0.078872,0.068615,NaN,mT5-base (gold),-,validation


## Konačan zaključak

**Poredak retrievera je stabilan kroz sve generatore i oba splita:** dense retrieval dosledno nadmašuje BM25 i TF-IDF na F1 i ROUGE-L, za Qwen3-8B, mT0-small, i (na validaciji) neuspeli mT5-base pokušaj. Ovaj zaključak se održao bez obzira na to koji je generator ili prompt korišćen.

**Izbor generatora utiče na kvalitet odgovora mnogo više nego izbor retrievera.** Razlika između Qwen3-8B (F1≈0.29-0.32) i mT0-small (F1≈0.14-0.17) je otprilike 2×, dok je razlika između najboljeg i najlošijeg retrievera unutar istog generatora svega ≈0.03-0.05. Za RAG sistem izgrađen na malom, fine-tuned generatoru, sam generator — ne retrieval — je glavno usko grlo kvaliteta finalnog odgovora.

**Dizajn prompta je bitan kod velikih instrukcijski-podešenih modela.** Prebacivanje Qwen3-8B sa opširnog na sažet sistemski prompt dosledno je poboljšalo F1/ROUGE-L kroz sva tri retrievera na test skupu, bez ikakvog ponovnog treniranja.

**Leksičke metrike (F1, ROUGE-L) sistematski kažnjavaju tačne, ali drugačije formulisane ili opširnije odgovore.** Ručno gledanje je više puta pokazalo slučajeve gde je generisani odgovor bio suštinski tačan, ali je dobio nizak skor zbog parafraziranja ili dodatnih detalja. Semantička sličnost (embedding cosine) je dodata kao dopunska metrika, iako su njene vrednosti kod Qwen-a bile zbijene i neinformativne za poređenje retrievera, dok je kod mT0-small pratila isti pad kvaliteta vidljiv u F1/ROUGE-L  — što sugeriše da je informativnija pri poređenju generatora vrlo različitog kvaliteta nego pri poređenju retrievera unutar istog generatora.

**Ponavljajući kvalitativni obrazac greške kroz sva tri generatora:** nijedan pouzdano ne reprodukuje konkretne imenovane činjenice iz referentnog odgovora (npr. "Therac-25", "Patriot", "Ariane 5") kada se traži konkretan primer, već daje uverljive ali generičke odgovore. Ovo ukazuje na retrieval ograničenje, ne samo na ograničenje generatora, i bio bi prirodan sledeći korak za istraživanje (npr. provera da li su ti chunkovi dosledno van top-k za sva tri retrievera). 

In [13]:
final_comparison_df.to_csv(
    EVALUATION_DIR / "final_comparison.csv",
    index=False,
)

print("Sačuvano:", EVALUATION_DIR / "final_comparison.csv")

Sačuvano: /home/anja/Desktop/MU/Student-Question-Answering-from-Course-Materials/data/evaluation/final_comparison.csv
